# Практическая работа № 5 (пример выполнения)
## State, Strategy, Command для «Тайны исчезнувшего дива»

**Курс:** ООП для разработки компьютерных игр  
**Модуль:** 5 — Паттерны: State, Strategy, Command  
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине  
**Формат:** демонстрационный блокнот с полным разбором задания  
**Цель:** показать, как добавить в проект FSM состояний, стратегии поведения Кузи и команды с отменой

---

## 📖 Контекст

После модулей 1–4 проект имеет:

| Слой | Что есть |
|---|---|
| **Абстракции** | `GameObject` (ABC), протоколы способностей |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue`, `Entity` + компоненты |
| **Системы** | Командные (`InvestigationSystem`, ...) и обновления (`MovementSystem`, ...) |
| **Оркестрация** | `Game`, `GameLoop`, `GameState` (хранилище) |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer` |

## 🚧 Какие проблемы остались

1. **Состояния игры смешаны с логикой.** `Game.run()` содержит `while` с вводом и проверкой победы. Меню, игра и победа — не формализованы.
2. **Поведение Кузи зашито в `Div`.** Если он должен вести себя агрессивно — правим класс.
3. **Действия игрока необратимы.** Игрок не может отменить ход или посмотреть историю.

## 🎯 Что демонстрирует блокнот

| Паттерн | Проблема | Улучшение |
|---|---|---|
| **State** | `if/elif` по состояниям | `GameState` FSM с `enter`/`update`/`exit` |
| **Strategy** | Поведение Кузи зашито | `BehaviorStrategy` — смена на лету |
| **Command** | Действия необратимы | `CommandHistory` с undo |

## Ключевая идея

Три паттерна решают три разные проблемы, но работают вместе: состояния управляют режимами, стратегии — поведением AI, команды — действиями игрока.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

import os
os.makedirs('mystic_detective/core', exist_ok=True)
print("✅ Готово")

---
# Шаг 1. Теория: зачем нужны три паттерна

## 1.1. State: состояния как объекты

**Проблема:** `if self.state == 'menu'`, `elif self.state == 'game'` — каждое новое состояние добавляет `elif`, логика размазана.

**Решение:** каждое состояние — класс с `enter`, `update`, `exit`. `Game` делегирует `update` текущему состоянию.

## 1.2. Strategy: алгоритмы как объекты

**Проблема:** поведение Кузи зашито в методы `Div`. Чтобы изменить — правим класс.

**Решение:** `BehaviorStrategy` — отдельный объект, который можно заменить на лету.

## 1.3. Command: действия как объекты

**Проблема:** действие игрока выполняется сразу и необратимо.

**Решение:** каждое действие — объект с `execute()` и `undo()`. `CommandHistory` хранит стек и позволяет отменять.

## 1.4. Как это улучшит проект

- **State:** меню, игра, победа — без `if/elif`.
- **Strategy:** Кузя может быть осторожным, агрессивным или игривым.
- **Command:** игрок может отменить ход, посмотреть историю.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html), [Strategy](https://gameprogrammingpatterns.com/strategy.html), [Command](https://gameprogrammingpatterns.com/command.html).
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state), [Strategy](https://refactoring.guru/design-patterns/strategy), [Command](https://refactoring.guru/design-patterns/command).

---
# Шаг 2. Реализация State: FSM игры

## 2.1. Абстрактный `GameState`

**Важно:** переименуем старое хранилище `GameState` (из модуля 3) в `WorldState`, чтобы не путать с FSM.

## 2.2. Состояния

- **`MenuState`** — заголовок, ожидание ввода.
- **`PlayingState`** — основной игровой цикл.
- **`VictoryState`** — победа.

**Ключевое:** `update` состояния может вызвать `game.change_state(...)` для перехода.

## 2.3. `Game.change_state`

```python
def change_state(self, new_state):
    self._state.exit(self)
    self._state = new_state
    self._state.enter(self)
```

Это гарантирует, что `exit` и `enter` вызываются в правильном порядке.

In [ ]:
with open('mystic_detective/core/game_states.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"GameState — FSM состояний игры.\n\n"""Заменяет if/elif по режимам игры.\n"""Каждое состояние — класс с enter/update/exit.\n""\"""\n\nfrom abc import ABC, abstractmethod\n\n\nclass GameState(ABC):\n    \"\"\"Абстрактное состояние игры.\"\"\"\n\n    @abstractmethod\n    def enter(self, game) -> None:\n        \"\"\"Вызывается при входе в состояние.\"\"\"\n\n    @abstractmethod\n    def update(self, game, dt: float) -> None:\n        \"\"\"Вызывается каждый кадр.\"\"\"\n\n    @abstractmethod\n    def exit(self, game) -> None:\n        \"\"\"Вызывается при выходе из состояния.\"\"\"\n\n\nclass MenuState(GameState):\n    \"\"\"Главное меню.\"\"\"\n\n    def enter(self, game) -> None:\n        game.ui.show_message(\"=== ТАЙНА ИСЧЕЗНУВШЕГО ДИВА ===\")\n        game.ui.show_message(\"1. Начать расследование\")\n        game.ui.show_message(\"2. Выйти\")\n\n    def update(self, game, dt: float) -> None:\n        choice = game.ui.get_input(\"Выбор: \")\n        if choice == \"1\":\n            game.change_state(PlayingState())\n        elif choice == \"2\":\n            game.stop()\n\n    def exit(self, game) -> None:\n        pass\n\n\nclass PlayingState(GameState):\n    \"\"\"Основное расследование.\"\"\"\n\n    def enter(self, game) -> None:\n        game.ui.show_message(\"Расследование начато.\")\n\n    def update(self, game, dt: float) -> None:\n        # Обновить мир\n        for system in game.update_systems:\n            system.update(game.world.entities, dt)\n\n        # Проверить победу\n        if game.victory_checker.is_victory(game.world):\n            game.change_state(VictoryState())\n            return\n\n        # Обработать ввод\n        action = game.input_handler.read_action()\n        if action is None:\n            return\n        if action.name == \"quit\":\n            game.stop()\n            return\n        game.dispatch(action)\n\n    def exit(self, game) -> None:\n        game.ui.show_message(\"Расследование завершено.\")\n\n\nclass VictoryState(GameState):\n    \"\"\"Победа.\"\"\"\n\n    def enter(self, game) -> None:\n        game.ui.show_message(\"\\nВы раскрыли тайну исчезновения Владимира!\")\n\n    def update(self, game, dt: float) -> None:\n        game.stop()\n\n    def exit(self, game) -> None:\n        pass\n''')
print("✅ core/game_states.py создан")

## 2.4. Демонстрация FSM

Создадим упрощённый `Game` с FSM и `FakeUI`, чтобы показать переходы.

In [ ]:
from core.game_states import MenuState, PlayingState, VictoryState


class FakeUI:
    def __init__(self, inputs):
        self._inputs = iter(inputs)
        self.messages = []

    def show_message(self, msg):
        self.messages.append(msg)
        print(msg)

    def get_input(self, prompt=""):
        try:
            return next(self._inputs)
        except StopIteration:
            return "2"  # выход по умолчанию


class MiniGame:
    def __init__(self):
        self.ui = FakeUI(["1", "quit"])
        self._state = MenuState()
        self._running = True
        self.update_systems = []
        self.victory_checker = type("V", (), {"is_victory": lambda self, w: False})()
        self.world = type("W", (), {"entities": []})()
        self.input_handler = type("I", (), {"read_action": lambda self: type("A", (), {"name": "quit", "args": ()})()})()

    def change_state(self, new_state):
        self._state.exit(self)
        self._state = new_state
        self._state.enter(self)

    def stop(self):
        self._running = False

    def dispatch(self, action):
        pass

    def run(self, frames=5):
        self._state.enter(self)
        for _ in range(frames):
            if not self._running:
                break
            self._state.update(self, 1/60)


print("=== FSM: Menu -> Playing -> Victory ===")
game = MiniGame()
game.run()
print(f"\nФинальное состояние: {type(game._state).__name__}")

**Что мы видим:** `MenuState` обработал ввод «1» и переключил на `PlayingState`. `PlayingState` получил команду «quit» и остановил игру. Переходы — через `game.change_state(...)`, без `if/elif` в `Game`.

---
# Шаг 3. Реализация Strategy: поведение Кузи

## 3.1. Протокол `BehaviorStrategy`

Три стратегии:
- **`CautiousStrategy`** — осторожный: избегает опасностей, возвращает `"hide"`.
- **`AggressiveStrategy`** — агрессивный: атакует, возвращает `"attack"`.
- **`PlayfulStrategy`** — игривый: случайные действия.

## 3.2. Интеграция с `Div`

`Div` получает стратегию через конструктор и может менять через `change_strategy`.

In [ ]:
with open('mystic_detective/core/strategies.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"BehaviorStrategy — стратегии поведения Кузи.\n\n"""Каждая стратегия возвращает решение: scout, hide, attack.\n""\"""\n\nfrom typing import Protocol, runtime_checkable\n\n\n@runtime_checkable\nclass BehaviorStrategy(Protocol):\n    \"\"\"Протокол стратегии поведения.\"\"\"\n\n    def decide(self, div, state) -> str:\n        \"\"\"Вернуть решение: scout, hide, attack, play.\"\"\"\n        ...\n\n\nclass CautiousStrategy:\n    \"\"\"Осторожная стратегия: прятаться.\"\"\"\n\n    def decide(self, div, state) -> str:\n        if div.loyalty < 30:\n            return \"hide\"\n        return \"scout\"\n\n\nclass AggressiveStrategy:\n    \"\"\"Агрессивная стратегия: атаковать.\"\"\"\n\n    def decide(self, div, state) -> str:\n        return \"attack\"\n\n\nclass PlayfulStrategy:\n    \"\"\"Игривая стратегия: случайные действия.\"\"\"\n\n    def __init__(self, seed: int = 42):\n        import random\n        self._rng = random.Random(seed)\n\n    def decide(self, div, state) -> str:\n        return self._rng.choice([\"scout\", \"play\", \"hide\"])\n''')
print("✅ core/strategies.py создан")

## 3.3. Демонстрация смены стратегии

In [ ]:
from core.strategies import CautiousStrategy, AggressiveStrategy, PlayfulStrategy


class MiniDiv:
    def __init__(self, name, loyalty=50):
        self.name = name
        self.loyalty = loyalty
        self._strategy = CautiousStrategy()

    def change_strategy(self, strategy):
        self._strategy = strategy

    def decide(self, state=None):
        return self._strategy.decide(self, state)


kuzia = MiniDiv("Кузя", loyalty=50)
print(f"Осторожный: {kuzia.decide()}")

kuzia.change_strategy(AggressiveStrategy())
print(f"Агрессивный: {kuzia.decide()}")

kuzia.change_strategy(PlayfulStrategy(seed=1))
print(f"Игривый: {kuzia.decide()}")

**Что мы видим:** поведение Кузи меняется на лету, без изменения класса `Div`. Это и есть Strategy.

---
# Шаг 4. Реализация Command: undo

## 4.1. Абстрактная `Command`

Методы: `execute()` и `undo()`.

## 4.2. `MoveCommand`

Хранит `from_location` и `to_location`. `execute` перемещает, `undo` возвращает.

## 4.3. `CommandHistory`

Стек выполненных команд.

In [ ]:
with open('mystic_detective/core/commands.py', 'w', encoding='utf-8') as f:
    f.write('''\"\"\"Command pattern с undo.\n\n"""Каждое действие — объект с execute и undo.\n"""CommandHistory хранит стек.\n""\"""\n\nfrom abc import ABC, abstractmethod\n\n\nclass Command(ABC):\n    \"\"\"Абстрактная команда.\"\"\"\n\n    @abstractmethod\n    def execute(self, world) -> None:\n        ...\n\n    @abstractmethod\n    def undo(self, world) -> None:\n        ...\n\n\nclass MoveCommand(Command):\n    \"\"\"Перемещение между локациями.\"\"\"\n\n    def __init__(self, from_location: str, to_location: str):\n        self.from_location = from_location\n        self.to_location = to_location\n\n    def execute(self, world) -> None:\n        world.current_location_name = self.to_location\n\n    def undo(self, world) -> None:\n        world.current_location_name = self.from_location\n\n\nclass CommandHistory:\n    \"\"\"История выполненных команд.\"\"\"\n\n    def __init__(self):\n        self._history = []\n\n    def execute(self, command: Command, world) -> None:\n        command.execute(world)\n        self._history.append(command)\n\n    def undo_last(self, world) -> bool:\n        if not self._history:\n            return False\n        command = self._history.pop()\n        command.undo(world)\n        return True\n\n    @property\n    def size(self) -> int:\n        return len(self._history)\n''')
print("✅ core/commands.py создан")

## 4.4. Демонстрация undo

In [ ]:
from core.commands import MoveCommand, CommandHistory


class MiniWorld:
    def __init__(self):
        self.current_location_name = "особняк"


world = MiniWorld()
history = CommandHistory()

print(f"Старт: {world.current_location_name}")

history.execute(MoveCommand("особняк", "академия"), world)
print(f"После move: {world.current_location_name}")

history.execute(MoveCommand("академия", "пустошь"), world)
print(f"После второго move: {world.current_location_name}")

history.undo_last(world)
print(f"После undo: {world.current_location_name}")

history.undo_last(world)
print(f"После второго undo: {world.current_location_name}")

**Что мы видим:** `CommandHistory` хранит стек, `undo_last` отменяет последнее действие. Игрок может вернуться на шаг назад.

---
# Шаг 5. Интеграция в `Game`

## 5.1. Что нужно изменить

- `Game` получает `current_state: GameState` (FSM).
- `Game.run()` делегирует `update` состоянию.
- `Game.change_state(new_state)` вызывает `exit`/`enter`.
- `Div` получает `BehaviorStrategy`.
- `Game` использует `CommandHistory` для действий.

## 5.2. Демонстрация: полный цикл

Объединим все три паттерна в упрощённой игре.

In [ ]:
from core.game_states import MenuState, PlayingState, VictoryState
from core.strategies import CautiousStrategy, AggressiveStrategy
from core.commands import MoveCommand, CommandHistory


class IntegratedGame:
    def __init__(self):
        self.ui = FakeUI(["1", "move", "академия", "undo", "quit"])
        self._state = MenuState()
        self._running = True
        self.world = type("W", (), {"current_location_name": "особняк", "entities": []})()
        self.update_systems = []
        self.victory_checker = type("V", (), {"is_victory": lambda self, w: False})()
        self.input_handler = self._make_input()
        self.history = CommandHistory()
        self.div = MiniDiv("Кузя")

    def _make_input(self):
        game = self
        class IH:
            def read_action(self):
                raw = game.ui.get_input("Выбор: ")
                if raw == "move":
                    target = game.ui.get_input("Куда: ")
                    return type("A", (), {"name": "move", "args": (target,)})
                if raw == "undo":
                    return type("A", (), {"name": "undo", "args": ()})
                if raw == "quit":
                    return type("A", (), {"name": "quit", "args": ()})
                return None
        return IH()

    def change_state(self, new_state):
        self._state.exit(self)
        self._state = new_state
        self._state.enter(self)

    def stop(self):
        self._running = False

    def dispatch(self, action):
        if action.name == "move":
            old = self.world.current_location_name
            self.history.execute(MoveCommand(old, action.args[0]), self.world)
            self.ui.show_message(f"Перемещение в {action.args[0]}")
        elif action.name == "undo":
            if self.history.undo_last(self.world):
                self.ui.show_message(f"Отмена. Теперь: {self.world.current_location_name}")
            else:
                self.ui.show_message("Нечего отменять")

    def run(self, frames=10):
        self._state.enter(self)
        for _ in range(frames):
            if not self._running:
                break
            self._state.update(self, 1/60)


print("=== Интеграция State + Strategy + Command ===")
game = IntegratedGame()
game.run()
print(f"\nФинальная локация: {game.world.current_location_name}")

**Что мы видим:**
- `MenuState` → `PlayingState` (State).
- В `PlayingState` выполняется `move` и `undo` (Command).
- Кузя может менять стратегию (Strategy).
Все три паттерна работают вместе.

---
# Шаг 6. Юнит-тесты

## 6.1. Тесты состояний
```python
def test_menu_transitions_to_playing():
    game = MiniGame()
    game._state = MenuState()
    game._state.enter(game)
    # ... проверить, что после ввода "1" состояние PlayingState
```

## 6.2. Тесты стратегий
```python
def test_aggressive_always_attack():
    div = MiniDiv("Кузя")
    div.change_strategy(AggressiveStrategy())
    assert div.decide() == "attack"
```

## 6.3. Тесты команд
```python
def test_undo_move():
    world = MiniWorld()
    history = CommandHistory()
    history.execute(MoveCommand("a", "b"), world)
    history.undo_last(world)
    assert world.current_location_name == "a"
```

Запустите `pytest -v` — все тесты должны проходить.

---
# Шаг 7. Документация

## 7.1. ADR-005

```markdown
# ADR-005: State, Strategy, Command

## Контекст
- Состояния игры через if/elif;
- Поведение Кузи зашито;
- Действия необратимы.

## Решение
- State: MenuState, PlayingState, VictoryState;
- Strategy: Cautious, Aggressive, Playful;
- Command: MoveCommand + CommandHistory.

## Альтернативы
1. Enum + if/elif — проще, но нарушает OCP.
2. Behavior tree — мощнее, но избыточно.

## Последствия
- ✅ Новое состояние = новый класс.
- ✅ Поведение Кузи меняется на лету.
- ✅ Undo работает.
- ⚠️ Больше классов.
```

## 7.2. UML
Добавьте диаграммы:
- State: `GameState` ← `MenuState`, `PlayingState`, `VictoryState`.
- Strategy: `BehaviorStrategy` ← `Cautious`, `Aggressive`, `Playful`.
- Command: `Command` ← `MoveCommand`; `CommandHistory` → `Command`.

## 7.3. PEP 8
```bash
pycodestyle core/game_states.py core/strategies.py core/commands.py
```
Исправьте замечания.

---
# Итоги

## Что сделали

| Паттерн | Проблема | Улучшение |
|---|---|---|
| **State** | `if/elif` по состояниям | FSM: `MenuState`, `PlayingState`, `VictoryState` |
| **Strategy** | Поведение Кузи зашито | `Cautious`, `Aggressive`, `Playful` — смена на лету |
| **Command** | Действия необратимы | `MoveCommand` + `CommandHistory` с undo |

## Что изменилось в проекте

**До:**
- `Game.run()` с вводом и проверкой победы;
- поведение Кузи в `Div`;
- действия необратимы.

**После:**
- `GameState` FSM с `enter`/`update`/`exit`;
- `BehaviorStrategy` для Кузи;
- `CommandHistory` с undo;
- все три паттерна интегрированы.

## Как это подготовит следующие модули

- **Модуль 6 (Observer, Factory):** события от состояний и команд пойдут через Observer; фабрики для создания состояний.
- **Модуль 7 (ECS):** состояния и стратегии станут компонентами.
- **Финальный проект:** три паттерна — основа архитектуры.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html), [Strategy](https://gameprogrammingpatterns.com/strategy.html), [Command](https://gameprogrammingpatterns.com/command.html).
- Gamma E. et al. «Design Patterns».

**Углублённая:**
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state), [Strategy](https://refactoring.guru/design-patterns/strategy), [Command](https://refactoring.guru/design-patterns/command).
- [patternsgameprog.com: Command pattern](https://patternsgameprog.com/discover-python-and-patterns-12-command-pattern).
- [Stack Overflow: Strategy vs State](https://stackoverflow.com/questions/1658192/what-is-the-difference-between-strategy-design-pattern-and-state-design-pattern).